# 18 — Fundraising Package Generator
> One company brief in. Three investor-ready packages out — each written in the language and priorities of a different audience: venture capital, private equity, and family office.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


# --- Data shapes ---

class FundraisingMaterials(BaseModel):
    target_persona: Literal["vc", "pe", "family_office"]
    investor_thesis: str = Field(
        description="What THIS investor type cares about, framed in their language"
    )
    headline_metrics: List[str] = Field(
        description="3-5 metrics most relevant to this persona (e.g. ARR, NRR for VC; EBITDA for PE)"
    )
    narrative_angle: str = Field(
        description="The framing and story that lands best with this specific audience"
    )
    key_asks: List[str] = Field(
        description="What you are asking for, framed specifically for this persona"
    )
    objection_responses: List[str] = Field(
        description="Pre-emptive responses to the top 2 likely objections from this persona"
    )
    suggested_materials: List[str] = Field(
        description="Which documents to send first to this persona and in what order"
    )


class FundraisingPackage(BaseModel):
    """Audience-targeted fundraising materials generated for three investor personas."""

    company: Optional[str] = None
    round_type: str = Field(description="The fundraising round, e.g. 'Series A'")
    vc_materials: FundraisingMaterials
    pe_materials: FundraisingMaterials
    family_office_materials: FundraisingMaterials
    universal_value_props: List[str] = Field(
        description="Strengths that resonate across all investor personas"
    )


# --- Persona-specific system prompts ---

PERSONA_PROMPTS = {
    "vc": (
        "You are drafting fundraising materials for a VENTURE CAPITAL investor. "
        "VCs want TAM, growth velocity, and founder-market fit. "
        "Speak in ARR, NRR, burn multiple, and CAC payback. "
        "Frame the story around category creation and long-term compounding."
    ),
    "pe": (
        "You are drafting fundraising materials for a PRIVATE EQUITY investor. "
        "PE firms want EBITDA (actual or near-term path), operational efficiency, "
        "and a clear exit multiple. Avoid hype. "
        "Frame the story around margin expansion, process improvement, and downside protection."
    ),
    "family_office": (
        "You are drafting fundraising materials for a FAMILY OFFICE investor. "
        "Family offices want capital preservation, dividend potential, "
        "and downside protection. They are long-horizon, relationship-driven. "
        "Frame the story around resilience, governance quality, and stable compounding."
    ),
}


# --- Agent logic ---

def _draft_for_persona(
    llm: ChatOpenAI,
    brief: str,
    persona: Literal["vc", "pe", "family_office"],
) -> FundraisingMaterials:
    system = SystemMessage(PERSONA_PROMPTS[persona])
    drafter = system | llm.with_structured_output(FundraisingMaterials)
    return drafter.invoke(
        HumanMessage(
            content=f"Company brief:\n\n{brief}\n\nDraft materials for target_persona='{persona}'"
        )
    )


def run(company_brief: str, round_type: str = "Series A") -> FundraisingPackage:
    """
    Takes a company brief and returns three investor-ready packages:
    one for VCs, one for PE firms, one for family offices.
    """
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    vc = _draft_for_persona(llm, company_brief, "vc")
    pe = _draft_for_persona(llm, company_brief, "pe")
    fo = _draft_for_persona(llm, company_brief, "family_office")

    return FundraisingPackage(
        round_type=round_type,
        vc_materials=vc,
        pe_materials=pe,
        family_office_materials=fo,
        universal_value_props=vc.headline_metrics[:3],
    )


print("Agent ready.")

## The scenario

Meridian Health Analytics sells clinical workflow software to mid-size hospital groups. They are raising a Series A with strong retention and gross margins but no path to EBITDA yet — a profile that reads very differently to a VC, a PE firm, and a family office. The agent will produce a tailored fundraising package for each audience: the thesis to lead with, the metrics that land, the narrative angle to build, and the objections to pre-empt before the meeting.

In [ ]:
BRIEF = """
COMPANY: Meridian Health Analytics Inc.

Business: B2B SaaS — clinical workflow and patient flow optimisation for mid-size hospital groups.
Founded: 2021  |  HQ: Austin, TX  |  Employees: 52

FINANCIAL PROFILE:
  ARR: USD 6.4m  (+81% YoY)
  Gross margin: 71%
  Net revenue retention: 122%
  Burn rate: USD 520k/month
  Runway: 11 months

TRANSACTION:
  Seeking Series A: USD 18m
  Use of proceeds: 8 enterprise sales hires + expand into outpatient clinic segment

MARKET:
  143 hospital customers (largest = 22% of ARR)
  Addressable market: US clinical workflow SaaS USD 4.1bn
  Differentiation: Real-time bed and staffing optimisation; 2 direct VC-backed competitors
  NRR 122%; annual churn <4%
  HIPAA-compliant infrastructure; SOC 2 Type II certified

TEAM:
  CEO (founder, ex-hospital operations director 10 years)
  CTO (co-founder, ex-Microsoft Health)
  VP Sales (hired 6 months ago, ex-Epic Systems)
  No CFO yet; fractional finance currently
"""

package = run(BRIEF, round_type="Series A")

company_name = package.company or "Meridian Health Analytics"

print("=" * 65)
print(f"FUNDRAISING PACKAGE | {company_name} | {package.round_type}")
print("=" * 65)

if package.universal_value_props:
    print("\nUNIVERSAL VALUE PROPS (resonate across all audiences):")
    for v in package.universal_value_props:
        print(f"  + {v}")


def print_persona(label: str, mat) -> None:
    print(f"\n{'=' * 50}")
    print(f"FOR: {label}")
    print(f"{'=' * 50}")
    print(f"\nThesis:\n  {mat.investor_thesis}")
    print("\nHeadline metrics:")
    for m in mat.headline_metrics:
        print(f"  - {m}")
    print(f"\nNarrative angle:\n  {mat.narrative_angle}")
    print("\nKey asks:")
    for a in mat.key_asks:
        print(f"  - {a}")
    print("\nObjection responses:")
    for o in mat.objection_responses:
        print(f"  - {o}")
    print("\nSend these materials first (in order):")
    for s in mat.suggested_materials:
        print(f"  - {s}")


print_persona("VENTURE CAPITAL", package.vc_materials)
print_persona("PRIVATE EQUITY", package.pe_materials)
print_persona("FAMILY OFFICE", package.family_office_materials)

## Use your own data

Replace the `BRIEF` string in the cell above with:
- Your company's financials, headcount, and transaction details
- The specific round type and use of proceeds you are raising for

The agent will return a full investor package for each audience — including the thesis to lead with, the metrics that land, the narrative to build, and the objections to pre-empt before the meeting.